# CriterionLens benchmarks-only job

This notebook runs KV-cache, tiled-attention, and speculative-decoding benchmarks only. Speculative decoding uses deterministic synthetic token prompts. It does not evaluate validation or test data.

In [ ]:
import os
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch
from packaging.version import Version

devices = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print({'devices': devices, 'torch': torch.__version__, 'architectures': torch.cuda.get_arch_list()})
if len(devices) < 2 or any('T4' not in name for name in devices):
    raise RuntimeError(f'This benchmark job requires two Tesla T4 GPUs; found {devices}')

candidates = sorted({
    path.parent
    for path in Path('/kaggle/input').rglob('requirements.txt')
    if (path.parent / 'pyproject.toml').is_file()
})
if len(candidates) != 1:
    raise RuntimeError(
        'Expected exactly one code dataset with requirements.txt and pyproject.toml; '
        f'found {candidates}'
    )
DATASET_DIR = candidates[0]
print(f'Found code dataset at: {DATASET_DIR}')
CODE_DIR = Path('/kaggle/working/code')
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
shutil.copytree(DATASET_DIR, CODE_DIR, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE_DIR / 'requirements.txt')],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'],
    cwd=CODE_DIR,
    check=True,
)

try:
    torchao_version = version('torchao')
except PackageNotFoundError:
    torchao_version = None
if torchao_version and Version(torchao_version) < Version('0.16.1'):
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)

OUT_DIR = Path('/kaggle/working/out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
os_env = os.environ.copy()
os_env['PYTHONPATH'] = str(CODE_DIR / 'src') + os.pathsep + os_env.get('PYTHONPATH', '')
print(f'Code directory: {CODE_DIR}; output directory: {OUT_DIR}')

In [ ]:
def benchmark_command(script, *args, output_name):
    output_dir = CODE_DIR / 'results' / 'bench' / output_name
    return [sys.executable, script, *args, '--output-dir', str(output_dir)]


benchmark_commands = [
    benchmark_command(
        'scripts/bench_kv.py', '--config', 'configs/bench.yaml', '--model', model,
        output_name=output_name,
    )
    for model, output_name in (
        ('1.5B', 'kv_1p5b'), ('0.5B', 'kv_0p5b'), ('tiny', 'kv_tiny')
    )
]
benchmark_commands.extend([
    benchmark_command(
        'scripts/bench_attention.py', '--config', 'configs/attention_bench.yaml',
        output_name='attention',
    ),
    benchmark_command(
        'scripts/bench_speculative.py',
        '--config', 'configs/speculative_bench.yaml',
        '--prompt-source', 'synthetic', '--num-prompts', '20',
        '--input-length', '24', '--draft', 'base_0.5B',
        output_name='speculative_base_0p5b',
    ),
    benchmark_command(
        'scripts/bench_speculative.py',
        '--config', 'configs/speculative_bench.yaml',
        '--prompt-source', 'synthetic', '--num-prompts', '20',
        '--input-length', '24', '--draft', 'finetuned_0.5B',
        output_name='speculative_finetuned_0p5b',
    ),
])
for command in benchmark_commands:
    subprocess.run(command, cwd=CODE_DIR, env=os_env, check=True)
    print(f'Completed benchmark: {command[1:3]}')

In [ ]:
import zipfile

archive_path = OUT_DIR / 'bench_results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    root = CODE_DIR / 'results' / 'bench'
    if not root.exists():
        raise FileNotFoundError(f'Benchmark output directory was not created: {root}')
    for path in root.rglob('*'):
        if path.is_file():
            archive.write(path, path.relative_to(CODE_DIR))
print(f'Packaged benchmark results at {archive_path}')